# YOLO11 Training Workflow

这个 notebook 用于在服务器上完成 YOLO11 训练。默认使用当前服务器的 `yolo11` 环境和 GPU。

使用前准备：
- 数据集已经按 `train/val/test` 划分
- `mydata.yaml` 已经写好
- `myyolo11n.yaml` 已经写好
- 训练目录里已经放好 `yolo11n.pt`，或者允许自动下载


In [ ]:
from pathlib import Path
import subprocess
import shlex
import os

WORKDIR = Path('/home/ubuntu/ultralytics')
DATA_CFG = 'mydata.yaml'
MODEL_CFG = 'myyolo11n.yaml'
PRETRAINED = 'yolo11n.pt'
EPOCHS = 300
IMGSZ = 640
BATCH = 16
WORKERS = 8
DEVICE = '0'
OPTIMIZER = 'SGD'

print('WORKDIR =', WORKDIR)


def run(cmd: str):
    print(f'\n$ {cmd}\n')
    completed = subprocess.run(cmd, shell=True, executable='/bin/bash', text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed


## 1. 环境检查


In [ ]:
run('source ~/miniconda3/etc/profile.d/conda.sh && conda run -n yolo11 python -c "import torch, ultralytics; print(torch.__version__); print(torch.cuda.is_available()); print(torch.version.cuda); print(ultralytics.__version__)"')
assert WORKDIR.exists(), f'WORKDIR not found: {WORKDIR}'
assert (WORKDIR / DATA_CFG).exists(), f'Data cfg not found: {WORKDIR / DATA_CFG}'
assert (WORKDIR / MODEL_CFG).exists(), f'Model cfg not found: {WORKDIR / MODEL_CFG}'
print('data cfg =', WORKDIR / DATA_CFG)
print('model cfg =', WORKDIR / MODEL_CFG)


## 2. 准备预训练权重

如果本地没有 `yolo11n.pt`，这一格会自动下载。


In [ ]:
weights = WORKDIR / PRETRAINED
if not weights.exists():
    run(f'cd {shlex.quote(str(WORKDIR))} && wget -c https://github.com/ultralytics/assets/releases/download/v8.3.0/{PRETRAINED}')
else:
    print('weights already exists:', weights)


## 3. 开始训练


In [ ]:
cmd = (
    f'source ~/miniconda3/etc/profile.d/conda.sh && '
    f'cd {shlex.quote(str(WORKDIR))} && '
    f'conda run -n yolo11 yolo '
    f'task=detect mode=train '
    f'model={shlex.quote(MODEL_CFG)} '
    f'pretrained={shlex.quote(PRETRAINED)} '
    f'data={shlex.quote(DATA_CFG)} '
    f'epochs={EPOCHS} imgsz={IMGSZ} device={DEVICE} '
    f'optimizer={OPTIMIZER} workers={WORKERS} batch={BATCH} '
    f'amp=True cache=False lr0=0.01'
)
run(cmd)


## 4. 查看训练产物


In [ ]:
runs_dir = WORKDIR / 'runs' / 'detect'
assert runs_dir.exists(), f'No runs directory found: {runs_dir}'
for p in sorted(runs_dir.glob('train*')):
    print('run =', p)
    best = p / 'weights' / 'best.pt'
    if best.exists():
        print('  best.pt =', best)
